# NB30: learn the dense CYP2D6 primary screen

This secondary experiment predicts the **observed negative log2 fold change**, including weak compounds whose potency was not measured. It tests whether a stronger screen learner can improve ordering beyond the fixed-pseudo-label NB24 ensemble. Screen agreement is a proxy, not proof of improved blind potency or leaderboard score.

Seven fits: five scaffold folds, one holdout model, one final blind-test model. Each stage saves to Drive and resumes completed prediction batches. The feature block is the existing frozen surrogate PCA200 plus public qHTS predictions. **Existing auxiliary screen predictions are excluded**, because their original construction used rows in this experiment's holdout.

The comparison is the mean of NB24 pseudo runs a/b/c/e, not the full v30 ensemble. A fixed 20% rank blend is diagnostic only. This notebook produces prediction arrays and metrics, **not a submission CSV**. Keep v30 as the submission fallback.

**Comparison limitation:** the existing NB24 predictions used auxiliary screen features whose construction included current holdout labels. NB30 itself excludes those features and holds out all screen labels for the evaluation compounds, but the dense-screen comparison with NB24 is descriptive, not a strictly clean head-to-head validation.


In [ ]:
RUN = 'a'
ISO = 'CYP2D6'
N_ESTIMATORS = 4
PRED_BATCH = 256
SEED = 42
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'
OUTDIR = '/content/drive/MyDrive/cyp_results'

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No NVIDIA GPU detected. A GPU runtime is recommended.')
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%pip -q install tabpfn==2.2.1


In [ ]:
import gc, hashlib, json, os, platform, tarfile, time, uuid
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import GroupKFold
from tabpfn import TabPFNRegressor

CACHE = Path('/content/nb30_cache')
OUT = Path(OUTDIR)
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
assert Path('/content/drive/MyDrive').exists(), 'Mount Drive first.'
NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy', 'qhts_tr.npy', 'qhts_te.npy']
with tarfile.open(PAYLOAD_FILE) as archive:
    # Extract only named ordinary payload files, without accepting arbitrary archive paths.
    for name in NEED:
        matches = [m for m in archive.getmembers() if m.name in (name, './'+name)]
        assert len(matches) == 1 and matches[0].isfile(), f'Missing or invalid payload member: {name}'
        (CACHE / name).write_bytes(archive.extractfile(matches[0]).read())

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy').astype(bool)
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=False)
PCA_FILE = OUT / 'nb24_pca_surrogate_adme.npz'
assert PCA_FILE.exists(), 'Run NB24 surrogate PCA preparation first; its saved PCA file is required.'
with np.load(PCA_FILE, allow_pickle=False) as p:
    PF, PF_TE = p['tr'], p['te']
Q, Q_TE = np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'qhts_te.npy')
X, X_TE = np.hstack([PF, Q]).astype(np.float32), np.hstack([PF_TE, Q_TE]).astype(np.float32)
N, NTE = len(meta), len(mete)
assert len(X) == N == len(hold) == len(G) and len(X_TE) == NTE
assert X.shape[1] == X_TE.shape[1] == 205 and PF.shape[1] == 200
assert np.isfinite(X).all() and np.isfinite(X_TE).all(), 'Unexpected nonfinite feature values.'
assert meta['Molecule_Name'].is_unique and mete['Molecule_Name'].is_unique
assert set(meta['Molecule_Name']).isdisjoint(set(mete['Molecule_Name']))
COL = f'{ISO}_pIC50_direct_inhibition'
Y = meta[COL].to_numpy(float)
SCREEN = -meta[f'{ISO}_log2fc'].to_numpy(float)
SCREEN_OBS, LABELLED = np.isfinite(SCREEN), np.isfinite(Y)
ISO_ALL = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
obs_any = np.isfinite(meta[[f'{i}_pIC50_direct_inhibition' for i in ISO_ALL]].to_numpy(float)).any(1)
dev_all = np.flatnonzero(obs_any & ~hold)
ho_all = np.flatnonzero(obs_any & hold)
assert len(dev_all) + len(ho_all) == N
assert set(G[dev_all]).isdisjoint(set(G[ho_all])), 'Holdout scaffold groups overlap development.'
FOLDS = list(GroupKFold(5).split(dev_all, groups=G[dev_all]))
for tr, va in FOLDS:
    assert set(G[dev_all[tr]]).isdisjoint(set(G[dev_all[va]]))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu': torch.set_num_threads(4)

BASE_FILES = [OUT / f'nb24_{r}_pseudo_{ISO}.npz' for r in ['a', 'b', 'c', 'e']]
for p in BASE_FILES: assert p.exists(), f'Comparator missing: {p.name}'
BASE = {}
for p in BASE_FILES:
    with np.load(p, allow_pickle=False) as z:
        assert np.array_equal(z['dev'], np.flatnonzero(LABELLED & ~hold))
        assert np.array_equal(z['ho'], np.flatnonzero(LABELLED & hold))
        assert z['oof_full'].shape == z['hp_full'].shape == (N,) and z['test'].shape == (NTE,)
        for key in ['oof_full', 'hp_full', 'test']:
            BASE.setdefault(key, []).append(z[key].copy())
BASE = {k: np.mean(v, axis=0) for k, v in BASE.items()}
assert np.isfinite(BASE['oof_full'][dev_all]).all() and np.isfinite(BASE['hp_full'][ho_all]).all() and np.isfinite(BASE['test']).all()

def file_sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def array_sha(a): return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
CONFIG = dict(notebook='nb30_dense_screen_v1', run=RUN, iso=ISO, n_estimators=N_ESTIMATORS,
              pred_batch=PRED_BATCH, seed=SEED, target='negative_observed_log2fc_no_clipping',
              features='frozen_surrogate_PCA200_plus_public_qHTS5_no_aux',
              fixed_rank_blend_weight=0.2, fit_mode='low_memory', memory_saving_mode=True,
              n_jobs=1, tabpfn=version('tabpfn'), torch=version('torch'), numpy=version('numpy'),
              sklearn=version('scikit-learn'), device=DEVICE, n_train=N, n_test=NTE,
              inputs={name:file_sha(CACHE/name) for name in NEED}, pca_sha=file_sha(PCA_FILE),
              comparator_sha={p.name:file_sha(p) for p in BASE_FILES}, x_sha=array_sha(X), xt_sha=array_sha(X_TE))
assert CONFIG['tabpfn'] == '2.2.1'
SIG = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
CONFIG['signature'] = SIG
PREFIX = f'nb30_{RUN}_{ISO}_screen'

def save_json_atomic(path, data):
    tmp = Path(str(path)+'.'+uuid.uuid4().hex+'.tmp')
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False)); os.replace(tmp, path)
def save_npz_atomic(path, **kw):
    tmp = Path(str(path)+'.'+uuid.uuid4().hex+'.tmp.npz')
    np.savez_compressed(tmp, **kw); os.replace(tmp, path)
config_path = OUT / f'nb30_{RUN}_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, 'Configuration/source data changed. Choose a new RUN letter.'
else: save_json_atomic(config_path, CONFIG)
print('Feature shape:', X.shape, '| observed screen:', int(SCREEN_OBS.sum()), '| device:', DEVICE)
print('Development / holdout:', len(dev_all), len(ho_all), '| seven model fits | signature:', SIG[:12])
print('Checkpoints are saved under', OUT)


In [ ]:
def checked_stage(path, train_rows, eval_rows):
    with np.load(path, allow_pickle=False) as z:
        assert str(z['signature'].item()) == SIG, f'Wrong configuration in {path.name}'
        assert np.array_equal(z['train_rows'], train_rows) and np.array_equal(z['eval_rows'], eval_rows), path.name
        pred = z['pred'].copy()
        assert pred.shape == (len(eval_rows),)
        return pred

def fit_stage(stage, train_rows, eval_rows, eval_x, seed):
    train_rows = np.asarray(train_rows, dtype=int)
    eval_rows = np.asarray(eval_rows, dtype=int)
    assert SCREEN_OBS[train_rows].all()
    assert len(eval_rows) == len(eval_x) and len(train_rows) > 20
    final = OUT / f'{PREFIX}_{stage}.npz'
    partial = OUT / f'{PREFIX}_{stage}_partial.npz'
    if final.exists():
        pred = checked_stage(final, train_rows, eval_rows)
        assert np.isfinite(pred).all()
        print(stage, 'already complete; skipped', flush=True)
        return pred
    pred = checked_stage(partial, train_rows, eval_rows) if partial.exists() else np.full(len(eval_rows), np.nan)
    model = None
    started = time.time()
    try:
        model = TabPFNRegressor(device=DEVICE, n_estimators=N_ESTIMATORS, random_state=seed,
                                ignore_pretraining_limits=True, fit_mode='low_memory',
                                memory_saving_mode=True, n_jobs=1)
        model.fit(X[train_rows], SCREEN[train_rows])
        for lo in range(0, len(eval_rows), PRED_BATCH):
            hi = min(lo+PRED_BATCH, len(eval_rows))
            if np.isfinite(pred[lo:hi]).all(): continue
            got = np.asarray(model.predict(eval_x[lo:hi]), dtype=float).reshape(-1)
            assert got.shape == (hi-lo,) and np.isfinite(got).all(), f'Invalid prediction at {stage} {lo}:{hi}'
            pred[lo:hi] = got
            save_npz_atomic(partial, signature=np.array(SIG), train_rows=train_rows, eval_rows=eval_rows, pred=pred)
            print(f'{stage}: {np.isfinite(pred).sum()}/{len(pred)} predictions saved; {(time.time()-started)/60:.1f} min', flush=True)
        assert np.isfinite(pred).all()
        save_npz_atomic(final, signature=np.array(SIG), train_rows=train_rows, eval_rows=eval_rows, pred=pred)
        # Retain the small partial file as an additional recovery artifact.
        return pred
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

OOF_FULL = np.full(N, np.nan)
HP_FULL = np.full(N, np.nan)
started_all = time.time()
for k, (tr, va) in enumerate(FOLDS):
    tr_compounds, ev = dev_all[tr], dev_all[va]
    rows = tr_compounds[SCREEN_OBS[tr_compounds]]
    assert not np.intersect1d(rows, ev).size and not hold[rows].any()
    OOF_FULL[ev] = fit_stage(f'f{k}', rows, ev, X[ev], SEED+k)
rows = dev_all[SCREEN_OBS[dev_all]]
assert not hold[rows].any()
HP_FULL[ho_all] = fit_stage('hold', rows, ho_all, X[ho_all], SEED+100)
# Only final blind predictions use screen labels from the holdout.
rows = np.flatnonzero(SCREEN_OBS)
TEST = fit_stage('test', rows, np.arange(NTE), X_TE, SEED+200)
assert np.isfinite(OOF_FULL[dev_all]).all() and np.isfinite(HP_FULL[ho_all]).all() and np.isfinite(TEST).all()
print('All stages complete in', round((time.time()-started_all)/60, 1), 'minutes this session')


In [ ]:
def correlation(y, p):
    ok = np.isfinite(y) & np.isfinite(p)
    if ok.sum() < 3 or np.std(y[ok]) == 0 or np.std(p[ok]) == 0: return None
    return float(spearmanr(y[ok], p[ok]).statistic)
def rank_percentile(p): return (rankdata(p) - 0.5) / len(p)
def rank_mix(base, screen, w=0.2): return (1-w)*rank_percentile(base) + w*rank_percentile(screen)
def auc(p, pos, neg):
    if not len(pos) or not len(neg): return None
    r = rankdata(np.r_[p[pos], p[neg]])
    return float((r[:len(pos)].sum()-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg)))
# Define clean blanks using only development-labelled screen values.
cut = float(np.percentile(SCREEN[~hold & LABELLED & SCREEN_OBS], 5))
CLEAN = ~LABELLED & SCREEN_OBS & (SCREEN < cut)
def metrics(pred, idx):
    screened = idx[SCREEN_OBS[idx]]
    blank = idx[SCREEN_OBS[idx] & ~LABELLED[idx]]
    labelled = idx[LABELLED[idx]]
    labelled_screen = idx[SCREEN_OBS[idx] & LABELLED[idx]]
    clean = idx[CLEAN[idx]]
    return dict(screen_all_rho=correlation(SCREEN[screened], pred[screened]),
                screen_blank_rho=correlation(SCREEN[blank], pred[blank]),
                screen_labelled_rho=correlation(SCREEN[labelled_screen], pred[labelled_screen]),
                labelled_potency_rho=correlation(Y[labelled], pred[labelled]),
                label_clean_blank_auc=auc(pred, labelled_screen, clean),
                n_screen=len(screened), n_blank=len(blank), n_labelled=len(labelled), n_clean_blank=len(clean))
BLEND_OOF = np.full(N, np.nan)
for _, va in FOLDS:
    ev = dev_all[va]
    BLEND_OOF[ev] = rank_mix(BASE['oof_full'][ev], OOF_FULL[ev])
BLEND_HP = np.full(N, np.nan)
BLEND_HP[ho_all] = rank_mix(BASE['hp_full'][ho_all], HP_FULL[ho_all])
BLEND_TEST = rank_mix(BASE['test'], TEST)
RESULTS = dict(config=CONFIG, clean_blank_screen_cut=cut,
               interpretation='Screen response is a proxy. NB24 mean a/b/c/e is the comparator, not full v30. Existing NB24 predictions used auxiliary screen features whose construction included current holdout labels. NB30 excludes those features and holds out evaluation screen labels, but the NB24 dense-screen comparison is descriptive, not a strictly clean head-to-head validation. No submission decision is automatic.',
               methods={})
for name, dv, hp in [('nb24_pseudo_mean_abce', BASE['oof_full'], BASE['hp_full']),
                     ('nb30_screen', OOF_FULL, HP_FULL), ('fixed_20pct_rank_blend', BLEND_OOF, BLEND_HP)]:
    per_fold = [metrics(dv, dev_all[va]) for _, va in FOLDS]
    keys = [key for key in per_fold[0] if not key.startswith('n_')]
    foldmean = {key:float(np.mean([f[key] for f in per_fold if f[key] is not None]))
                if any(f[key] is not None for f in per_fold) else None for key in keys}
    RESULTS['methods'][name] = dict(dev_fold_mean=foldmean, dev_folds=per_fold,
                                   dev_pooled=metrics(dv, dev_all), holdout=metrics(hp, ho_all))
    print(name, '| dev fold mean:', foldmean, '| holdout:', RESULTS['methods'][name]['holdout'], flush=True)
dv_lab, ho_lab = np.flatnonzero(LABELLED & ~hold), np.flatnonzero(LABELLED & hold)
arm_path = OUT / f'{PREFIX}_arm.npz'
save_npz_atomic(arm_path, signature=np.array(SIG), oof_full=OOF_FULL, hp_full=HP_FULL, test=TEST,
                oof=OOF_FULL[dv_lab], dev=dv_lab, hold=HP_FULL[ho_lab], ho=ho_lab,
                dev_all=dev_all, ho_all=ho_all, blend_oof_full=BLEND_OOF,
                blend_hp_full=BLEND_HP, blend_test=BLEND_TEST)
results_path = OUT / f'results_nb30_{RUN}.json'
save_json_atomic(results_path, RESULTS)
print('Saved:', arm_path.name, 'and', results_path.name)
print('Do not upload screen predictions as pIC50. Review dense screen and labelled potency metrics together before constructing any candidate.')


In [ ]:
import zipfile
archive_path = Path(f'/content/nb30_{RUN}_results.zip')
mine = [OUT / f'nb30_{RUN}_config.json', OUT / f'results_nb30_{RUN}.json', OUT / f'{PREFIX}_arm.npz']
mine += [OUT / f'{PREFIX}_{stage}.npz' for stage in ['f0', 'f1', 'f2', 'f3', 'f4', 'hold', 'test']]
assert all(p.exists() for p in mine), 'Some stages are incomplete.'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in mine: z.write(p, p.name)
from IPython.display import FileLink, display
display(FileLink(str(archive_path)))
print('The same results are already saved to Drive.')
